In [1]:
import copy
import tempfile
import os
import json

from agent_demo import (
    INITIAL_STATE,
    SUMMARY_AGENT,
    CHAT_TURN_AGENT,
    run_audio_demo,
)

/home/m.gromadzki/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3966.24it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: ../../biobert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 

In [ ]:
for event in SUMMARY_AGENT.stream(INITIAL_STATE):
    for node_name, node_output in event.items():

        # Update state live
        print(node_name, node_output)

In [1]:
tmp = "[EXAM TYPE] CT chest PE protocol {period} [INDICATION] 54-year-old female, shortness of breath, evaluate for PE {period}TECchHNIQe[TECHNIQUE] Standard protocol {period} [FINDINGS] {colon} Pulmonary vasculature {colon} The main PA is patent {period} There are filling defects in the segmental.segmental branches of the right lower lobe {comma} compatible with acute PE {period} No saddle embolus {period} Lungs {colon} No pneumothorax.othorax {period} Small bilateral effusions {comma} right greater than left {period} {new paragraph} [IMPRESSION] {colon}"

In [2]:
import re

def normalize_medasr(text: str) -> str:
    # 1. Normalize special tokens
    replacements = {
        "{period}": ".",
        "{comma}": ",",
        "{colon}": ":",
        "{new paragraph}": "\n\n",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    # 2. Fix glued section headers like "TECchHNIQe[TECHNIQUE]"
    text = re.sub(r"[A-Za-z]+\[([A-Z ]+)\]", r"[\1]", text)

    # 3. Remove accidental duplicated fragments like:
    # segmental.segmental → segmental
    # pneumothorax.othorax → pneumothorax
    text = re.sub(r"\b(\w+)\.\1\b", r"\1", text)

    # 4. Fix partial word repetition after period (pneumothorax.othorax)
    text = re.sub(r"\b(\w+)\.(\w+)\b", 
                  lambda m: m.group(1) if m.group(2) in m.group(1) else m.group(0),
                  text)

    # 5. Add newline before section headers
    text = re.sub(r"\s*(\[[A-Z ]+\])", r"\n\1", text)

    # 6. Normalize spacing
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\n\s+", "\n", text)

    # 7. Clean punctuation spacing
    text = re.sub(r"\s+([.,:])", r"\1", text)
    text = re.sub(r"([.,:])(?=[^\s])", r"\1 ", text)

    return text.strip()

In [3]:
normalize_medasr(tmp)

'[EXAM TYPE] CT chest PE protocol. [INDICATION] 54-year-old female, shortness of breath, evaluate for PE. [TECHNIQUE] Standard protocol. [FINDINGS]: Pulmonary vasculature: The main PA is patent. There are filling defects in the segmental branches of the right lower lobe, compatible with acute PE. No saddle embolus. Lungs: No pneumothorax. Small bilateral effusions, right greater than left. [IMPRESSION]:'